# 00 · Preparación del entorno y descarga de datos

**Proyecto:** ¿Lo que funciona afuera funciona en la APS chilena? Transferibilidad de estrategias de NER clínico.
**Curso:** IA Generativa y Modelos Fundacionales (prof. Rodrigo Salas) · **Equipo:** Sebastián Inostroza, Maite Bahamondes, Ruth Vergara

---

### Para qué sirve este notebook

Antes de entrenar o probar cualquier modelo hay que asegurarse de tres cosas:

1. Que el **entorno** (Python y sus librerías) está bien instalado y que el computador puede usar su aceleración gráfica.
2. Que los **modelos de lenguaje locales** (los LLM que corren en el propio computador, sin enviar datos a internet) están disponibles.
3. Que los **datos** están descargados y completos.

Este notebook revisa las tres cosas, paso a paso. Si alguien externo quiere reproducir el proyecto, este es el punto de partida.

> **Requisito previo:** haber corrido una vez `bash scripts/setup_mac.sh` en la terminal, y en VS Code haber elegido el kernel **.venv** (arriba a la derecha del notebook).

## 1. Ubicar el proyecto

La primera celda le dice a Python dónde está la carpeta del proyecto, para poder importar nuestro propio código (carpeta `src/`).

In [ ]:
from pathlib import Path
import sys

# Raíz del proyecto: funciona si abres el notebook desde VS Code en la carpeta notebooks/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
print("Raíz del proyecto:", RAIZ)

## 2. Revisar Python y las librerías principales

Usamos tres familias de herramientas:

| Librería | Para qué |
|---|---|
| `torch` (PyTorch) | El motor que entrena redes neuronales (Unidad 2 del curso) |
| `transformers` (Hugging Face) | Descargar y ajustar modelos tipo BERT (experimento E1) |
| `ollama` | Conversar con LLM locales desde Python (experimentos E2 y E4) |

La celda intenta importar cada una e informa si falta alguna.

In [ ]:
import importlib, platform

print("Python", platform.python_version(), "|", platform.platform())
for nombre in ["numpy", "pandas", "torch", "transformers", "datasets", "seqeval", "ollama", "yaml"]:
    try:
        m = importlib.import_module(nombre)
        print(f"  OK   {nombre:<13} {getattr(m, '__version__', '')}")
    except ImportError:
        print(f"  FALTA {nombre:<12} -> pip install -r requirements.txt")

## 3. ¿Puede el Mac usar su GPU?

Los Mac con chip Apple (M1–M5) tienen una aceleración llamada **MPS** (*Metal Performance Shaders*). Entrenar un modelo BERT con MPS puede ser varias veces más rápido que solo con el procesador. Si aparece `False`, el código igual funciona, pero más lento.

In [ ]:
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("MPS disponible:", torch.backends.mps.is_available())
    dispositivo = "mps" if torch.backends.mps.is_available() else "cpu"
    x = torch.randn(512, 512, device=dispositivo)
    print("Prueba de cálculo en", dispositivo, "->", float((x @ x).mean()))
except ImportError:
    print("PyTorch no está instalado en este kernel.")

## 4. ¿Están los LLM locales?

**Ollama** es un programa que descarga y ejecuta LLM en el computador. Los modelos que usamos (familias Llama y Qwen, de 7–8 mil millones de parámetros) están **cuantizados**: sus números se guardan con menos precisión para que quepan en la memoria de un Mac de 24 GB, con una pérdida de exactitud pequeña.

La celda lista los modelos instalados y les hace una pregunta corta para comprobar que responden. Que el modelo corra localmente importa: con la Ley 21.719, los datos clínicos reales no deberían enviarse a servicios externos.

In [ ]:
import yaml

cfg = yaml.safe_load(open(RAIZ / "configs" / "experimentos.yaml"))
print("Modelos configurados:", cfg["llm_locales"])
try:
    import ollama
    instalados = [m.model for m in ollama.list().models]
    print("Instalados en Ollama:", instalados)
    modelo = cfg["llm_locales"][0]
    r = ollama.chat(model=modelo, messages=[{"role": "user", "content": "¿Qué significa HTA en una ficha clínica chilena? Responde en 5 palabras."}],
                    options={"temperature": 0})
    print(f"\n{modelo} responde:", r.message.content)
except Exception as e:
    print("Ollama no disponible todavía:", e)
    print("Instálalo desde https://ollama.com/download y corre: ollama pull llama3.1:8b")

## 5. Descargar los datos

Los corpus son públicos. El script `src/datos/descargar.py` los baja a la carpeta `data/raw/`, que **no se sube a GitHub** (así lo dice el archivo `.gitignore`). Esto respeta la licencia no comercial del corpus y evita subir datos clínicos, aunque estén anonimizados.

| Corpus | Qué es | Rol en el proyecto |
|---|---|---|
| **CWLC** | 5.000 derivaciones de APS chilena anotadas por médicos (7 tipos de entidad) | Entrenamiento y examen principal |
| **DisTEMIST** | 1.000 casos clínicos de España con enfermedades anotadas | Contraste: texto "de manual" |
| **GES** | ~941 derivaciones clasificadas GES / no GES por expertos | Tarea opcional |

Si un archivo ya existe, no se vuelve a descargar.

In [ ]:
import subprocess

resultado = subprocess.run([sys.executable, str(RAIZ / "src" / "datos" / "descargar.py"), "cwlc"],
                           capture_output=True, text=True)
print(resultado.stdout or resultado.stderr)

## 6. Verificar que el corpus está completo

El CWLC viene en **formato CoNLL**: una palabra por línea, seguida de su etiqueta (`B-Disease` = empieza una enfermedad, `I-Disease` = continúa, `O` = nada), y una línea en blanco entre oraciones. Hay un archivo por tipo de entidad, porque una misma palabra puede pertenecer a dos entidades a la vez (por ejemplo "cáncer de **mama**": enfermedad y parte del cuerpo).

Nuestro módulo `datos.cwlc` junta los 7 archivos de cada partición (*split*). Esperamos **8.014 / 890 / 990** oraciones en train / dev / test.

In [ ]:
from datos.cwlc import cargar_split, resumen
import pandas as pd

carpeta = RAIZ / "data" / "raw" / "cwlc"
tabla = pd.DataFrame({s: resumen(cargar_split(carpeta, s)) for s in ["train", "dev", "test"]})
tabla

## 7. Verificar las métricas

Antes de medir modelos, comprobamos que nuestra calculadora de métricas funciona. Las pruebas automáticas (`tests/`) revisan, entre otras cosas, el ejemplo de la guía: 10 entidades reales, el modelo marca 8 y acierta 6, lo que da un **F1 ≈ 0,67**.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", str(RAIZ / "tests")], capture_output=True, text=True)
print(r.stdout[-400:])

## Resumen

Si todas las celdas anteriores funcionaron:

- El entorno está listo y el Mac puede usar su GPU.
- Los LLM locales responden.
- El CWLC está descargado y completo.
- Las métricas están verificadas.

**Siguiente notebook:** `01_exploracion_cwlc.ipynb`, para conocer el corpus antes de usarlo.